# MNIST Digit Recognition

Unified notebook for loading, training, and evaluating a **Residual CNN with
Squeeze-and-Excitation attention** on the MNIST dataset.
Training is logged to TensorBoard; post-training analysis covers confusion matrices,
per-class precision / recall / F1, confidence distributions, and misclassification grids.

In [ ]:
%load_ext tensorboard

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import trange, tqdm

from load_mnist import build_mnist_dataloaders
from model import CNN
from train import train_epoch, evaluate, save_model_state, get_device
from evaluate import evaluate_model

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────
DATA_DIR = Path("../data").resolve()
LOG_DIR = Path("../runs").resolve()
MODEL_PATH = Path("../models/cnn.zip").resolve()

# ── Hyperparameters ─────────────────────────────────────────────
BATCH_SIZE = 256
EPOCHS = 100
LR = 3e-4  # AdamW base LR
MAX_LR = 1e-3  # OneCycleLR peak LR
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()
TTA_PASSES = 10  # test-time augmentation passes at evaluation

# ── Device ──────────────────────────────────────────────────────
device = get_device("auto")
print(f"Device : {device}")
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name}")
    print(f"VRAM   : {p.total_memory / 1e9:.1f} GB")

## Architecture

The model is a **Residual CNN with Squeeze-and-Excitation channel attention** (~2.8 M parameters):

| Stage | Blocks | Channels | Spatial output |
|---|---|---|---|
| Stem | Conv 3×3 → BN → ReLU | 1→64 | 28×28 |
| Stage 1 | ResBlock×2 + SEBlock | 64 | 14×14 (MaxPool) |
| Stage 2 | ResBlock×2 + SEBlock | 64→128 | 7×7 (MaxPool) |
| Stage 3 | ResBlock×2 + SEBlock | 128→256 | 1×1 (GlobalAvgPool) |
| Head | Dropout(0.5) → Linear | 256→10 | — |

Each **ResBlock** has two 3×3 conv layers with BN and a skip-connection shortcut
(1×1 projection when channels differ).
Each **SEBlock** squeezes to C/16, applies ReLU, then rescales channel responses
via a sigmoid gate.

In [ ]:
model = CNN().to(device)

rows = [
    {"Layer": n, "Shape": str(list(p.shape)), "Parameters": p.numel()}
    for n, p in model.named_parameters()
    if p.requires_grad
]
param_df = pd.DataFrame(rows)
print(f"Total trainable parameters: {param_df['Parameters'].sum():,}")
param_df.style.format({"Parameters": "{:,}"})

## Data

In [ ]:
train_loader, test_loader = build_mnist_dataloaders(
    root=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    download=False,
)
print(f"Train : {len(train_loader.dataset):,} samples  ({len(train_loader):,} batches)")
print(f"Test  : {len(test_loader.dataset):,} samples  ({len(test_loader):,} batches)")

In [ ]:
CLASS_NAMES = [str(i) for i in range(10)]

train_labels_all = np.array([int(y) for _, y in train_loader.dataset])
test_labels_all = np.array([int(y) for _, y in test_loader.dataset])

classes, train_counts = np.unique(train_labels_all, return_counts=True)
_, test_counts = np.unique(test_labels_all, return_counts=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, counts, split in zip(axes, [train_counts, test_counts], ["Train", "Test"]):
    bars = ax.bar(classes, counts, color="steelblue", edgecolor="white", linewidth=0.6)
    ax.set_xticks(classes)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_xlabel("Digit class")
    ax.set_ylabel("Count")
    ax.set_title(f"Class distribution — {split} set")
    for bar, c in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30,
            str(c),
            ha="center",
            va="bottom",
            fontsize=7,
        )
plt.tight_layout()
plt.show()

In [ ]:
# Show 8 sample images per digit class
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

indices_per_class = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(train_loader.dataset):
    c = int(label)
    if len(indices_per_class[c]) < 8:
        indices_per_class[c].append(idx)
    if all(len(v) == 8 for v in indices_per_class.values()):
        break

fig, axes = plt.subplots(10, 8, figsize=(10, 14))
fig.suptitle("Sample images — 8 per class", fontsize=11, y=1.005)
for row, cls in enumerate(range(10)):
    for col, si in enumerate(indices_per_class[cls]):
        img_t, _ = train_loader.dataset[si]
        img = np.clip(img_t.squeeze().numpy() * MNIST_STD + MNIST_MEAN, 0, 1)
        ax = axes[row][col]
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(
                f"Digit {cls}", rotation=0, labelpad=30, va="center", fontsize=9
            )
plt.tight_layout()
plt.show()

## Training

In [ ]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    div_factor=25.0,
    final_div_factor=1e4,
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
history = {"train_loss": [], "train_acc": [], "eval_loss": [], "eval_acc": []}

epoch_bar = trange(1, EPOCHS + 1, desc="Training", unit="epoch")
for epoch in epoch_bar:
    tr_loss, tr_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, scheduler
    )
    ev_loss, ev_acc = evaluate(model, test_loader, criterion, device)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["eval_loss"].append(ev_loss)
    history["eval_acc"].append(ev_acc)

    writer.add_scalar("cnn/Train/Loss", tr_loss, epoch)
    writer.add_scalar("cnn/Train/Accuracy", tr_acc, epoch)
    writer.add_scalar("cnn/Eval/Loss", ev_loss, epoch)
    writer.add_scalar("cnn/Eval/Accuracy", ev_acc, epoch)

    epoch_bar.set_postfix(
        tr_loss=f"{tr_loss:.4f}",
        tr_acc=f"{tr_acc:.4f}",
        ev_loss=f"{ev_loss:.4f}",
        ev_acc=f"{ev_acc:.4f}",
    )

writer.close()
save_model_state(model, MODEL_PATH)
print(f"Model saved  →  {MODEL_PATH}")
print(f"Best eval acc: {max(history['eval_acc']) * 100:.4f}%")

## TensorBoard

In [ ]:
%tensorboard --logdir ../runs

## Evaluation

In [ ]:
# Collect all test-set predictions, probabilities, and input images
model.eval()
all_targets, all_predictions, all_probs, all_images = [], [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Collecting predictions"):
        logits = model(images.to(device))
        probs = F.softmax(logits, dim=1).cpu()
        all_targets.append(labels)
        all_predictions.append(probs.argmax(dim=1))
        all_probs.append(probs)
        all_images.append(images)

all_targets = torch.cat(all_targets).numpy()
all_predictions = torch.cat(all_predictions).numpy()
all_probs = torch.cat(all_probs).numpy()
all_images = torch.cat(all_images)

In [ ]:
correct = int((all_targets == all_predictions).sum())
total = len(all_targets)
accuracy = correct / total

summary_df = pd.DataFrame(
    [
        {"Metric": "Correct / Total", "Value": f"{correct:,} / {total:,}"},
        {"Metric": "Overall accuracy", "Value": f"{accuracy * 100:.4f}%"},
        {"Metric": "Error rate", "Value": f"{(1 - accuracy) * 100:.4f}%"},
    ]
)
summary_df.style.hide(axis="index")

### Confusion Matrix

In [ ]:
NUM_CLASSES = 10


def compute_cm(t, p, n=NUM_CLASSES):
    cm = np.zeros((n, n), dtype=np.int64)
    for ti, pi in zip(t, p):
        cm[ti, pi] += 1
    return cm


cm = compute_cm(all_targets, all_predictions)

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(np.log2(cm + 1), cmap="Blues")
plt.colorbar(im, ax=ax, label="log₂(count + 1)")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix (log-scaled colour, raw counts annotated)")
threshold = cm.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=8,
            color="white" if cm[i, j] > threshold else "black",
        )
plt.tight_layout()
plt.show()

### Per-class Metrics

In [ ]:
rows = []
for c in range(NUM_CLASSES):
    tp = int(cm[c, c])
    fp = int(cm[:, c].sum() - tp)
    fn = int(cm[c, :].sum() - tp)
    support = int(cm[c, :].sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    rows.append(
        {
            "Digit": c,
            "Precision": round(prec, 6),
            "Recall": round(rec, 6),
            "F1": round(f1, 6),
            "Accuracy": round(tp / support if support else 0, 6),
            "Support": support,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }
    )

mdf = pd.DataFrame(rows)
macro = {
    "Digit": "macro avg",
    "Precision": round(mdf.Precision.mean(), 6),
    "Recall": round(mdf.Recall.mean(), 6),
    "F1": round(mdf.F1.mean(), 6),
    "Accuracy": round(mdf.Accuracy.mean(), 6),
    "Support": int(mdf.Support.sum()),
    "TP": "",
    "FP": "",
    "FN": "",
}
mdf = pd.concat([mdf, pd.DataFrame([macro])], ignore_index=True)
mdf.style.format({c: "{:.4f}" for c in ["Precision", "Recall", "F1", "Accuracy"]}).hide(
    axis="index"
)

### Confidence Distribution

In [ ]:
max_probs = all_probs.max(axis=1)
is_correct = all_targets == all_predictions

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 51)
ax.hist(
    max_probs[is_correct],
    bins=bins,
    alpha=0.65,
    color="steelblue",
    label=f"Correct ({is_correct.sum():,})",
    density=True,
)
ax.hist(
    max_probs[~is_correct],
    bins=bins,
    alpha=0.65,
    color="tomato",
    label=f"Misclassified ({(~is_correct).sum():,})",
    density=True,
)
ax.axvline(
    max_probs[is_correct].mean(),
    color="steelblue",
    ls="--",
    lw=1.2,
    label=f"Mean correct:  {max_probs[is_correct].mean():.3f}",
)
ax.axvline(
    max_probs[~is_correct].mean(),
    color="tomato",
    ls="--",
    lw=1.2,
    label=f"Mean wrong:    {max_probs[~is_correct].mean():.3f}",
)
ax.set_xlabel("Max softmax probability (model confidence)")
ax.set_ylabel("Density")
ax.set_title("Confidence Distribution — Correct vs. Misclassified")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Misclassified Samples

In [ ]:
# Most-confidently misclassified images
mask = all_targets != all_predictions
mis_idx = np.where(mask)[0]
order = mis_idx[np.argsort(-all_probs[mis_idx, all_predictions[mis_idx]])]

n_show = min(40, len(order))
cols = 8
n_rows = (n_show + cols - 1) // cols
conf_disp = all_probs[order[:n_show], all_predictions[order[:n_show]]]

fig, axes = plt.subplots(
    n_rows, cols, figsize=(cols * 1.4, n_rows * 1.9), squeeze=False
)
fig.suptitle(
    f"Most Confidently Misclassified — {n_show} of {len(order)} errors shown",
    fontsize=10,
    y=1.01,
)
for k, (idx, conf) in enumerate(zip(order[:n_show], conf_disp)):
    ax = axes[k // cols][k % cols]
    img = np.clip(all_images[idx].squeeze().numpy() * MNIST_STD + MNIST_MEAN, 0, 1)
    ax.imshow(img, cmap="gray")
    ax.set_title(
        "T:%d P:%d\n%.2f" % (all_targets[idx], all_predictions[idx], conf), fontsize=7
    )
    ax.axis("off")
for k in range(n_show, n_rows * cols):
    axes[k // cols][k % cols].axis("off")
plt.tight_layout()
plt.show()

### Per-class Accuracy

In [ ]:
pca = np.diag(cm) / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["tomato" if a < 0.99 else "steelblue" for a in pca]
bars = ax.bar(range(NUM_CLASSES), pca * 100, color=colors, edgecolor="white")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_xlabel("Digit class")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-class Accuracy on Test Set  (red = below 99%)")
ax.set_ylim(max(0, pca.min() * 100 - 1.5), 100.5)
ax.axhline(
    accuracy * 100,
    color="black",
    ls="--",
    lw=1,
    label=f"Overall: {accuracy * 100:.2f}%",
)
ax.legend(fontsize=9)
for bar, acc in zip(bars, pca):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{acc * 100:.2f}%",
        ha="center",
        va="bottom",
        fontsize=7,
    )
plt.tight_layout()
plt.show()

### Top-K Accuracy

In [ ]:
def topk_accuracy(probs, targets, k):
    topk = np.argsort(probs, axis=1)[:, -k:]
    return float(np.any(topk == targets[:, None], axis=1).mean())


topk_df = pd.DataFrame(
    [
        {
            "K": f"Top-{k}",
            "Accuracy": f"{topk_accuracy(all_probs, all_targets, k) * 100:.4f}%",
        }
        for k in range(1, 4)
    ]
)
topk_df.style.hide(axis="index")

### Test-Time Augmentation (TTA)

In [ ]:
single_acc = evaluate_model(model, test_loader, device, tta_passes=1)
tta_acc = evaluate_model(model, test_loader, device, tta_passes=TTA_PASSES)

print(f"Single-pass : {single_acc:.2f}%")
print(f"TTA {TTA_PASSES:>2}x      : {tta_acc:.2f}%")

tta_df = pd.DataFrame(
    [
        {"Mode": "Single pass", "Accuracy (%)": f"{single_acc:.4f}"},
        {"Mode": f"TTA  ({TTA_PASSES} passes)", "Accuracy (%)": f"{tta_acc:.4f}"},
        {"Mode": "Gain", "Accuracy (%)": f"{tta_acc - single_acc:+.4f} pp"},
    ]
)
tta_df.style.hide(axis="index")